# 02 — Metrics

Explores the metric registry and shows how to compute individual metrics directly — useful before the Runner is available (Phase 4).

Covers:
- Inspecting the registry
- `resolve_metric_type` — inferring `MetricType` from a metric name
- Computing performance metrics (classification + regression)
- Computing drift metrics — distribution distance (PSI, Wasserstein, MMD), statistical tests (KS, t-test, Mann-Whitney, Levene, CvM), and the chi-square test for categorical features, and target drift for P(Y) shift detection
- Reading `effect_size` and `effect_size_label` to avoid false alerts on large samples
- Computing statistics metrics (mean, std, quantile, top_category)
- `compute_status` — evaluating pass/fail against a threshold

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Synthetic data

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 300

df_ref = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_pred_proba": rng.uniform(0.1, 0.9, size=n),
        "age": rng.normal(40, 10, size=n),
        "income": rng.normal(60_000, 15_000, size=n),
        "region": rng.choice(["A", "B", "C"], size=n),
    }
)

# current window: age distribution shifted (simulating drift)
df_cur = pd.DataFrame(
    {
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_pred_proba": rng.uniform(0.1, 0.9, size=n),
        "age": rng.normal(50, 12, size=n),  # shifted mean
        "income": rng.normal(61_000, 14_000, size=n),
        "region": rng.choice(["A", "B", "C", "D"], size=n),  # new category
    }
)

print("Reference:", df_ref.shape, "  Current:", df_cur.shape)

Reference: (300, 6)   Current: (300, 6)


## 2. Metric registry

In [3]:
from ayn_ml.metrics.registry import _REGISTRY, get_metric, resolve_metric_type

print(f"{len(_REGISTRY)} metrics registered:\n")
for name, cls in sorted(_REGISTRY.items()):
    print(f"  {name:<20}  type={cls.metric_type.value}")

38 metrics registered:

  accuracy              type=performance
  auc                   type=performance
  aucpr                 type=performance
  brier                 type=performance
  cbpe_accuracy         type=performance
  cbpe_auc              type=performance
  cbpe_f1               type=performance
  cbpe_precision        type=performance
  cbpe_recall           type=performance
  chisquare             type=drift
  count                 type=statistics
  cramervonmises        type=drift
  demographic_parity    type=fairness
  disparate_impact      type=fairness
  equalized_odds        type=fairness
  f1                    type=performance
  ks_2samp              type=drift
  kurtosis              type=statistics
  levene                type=drift
  log_loss              type=performance
  mae                   type=performance
  mannwhitney           type=drift
  mape                  type=performance
  mean                  type=statistics
  median                type=stati

In [4]:
from ayn_ml.core.spec import MetricSpec

# metric_type resolved from registry — no need to set it explicitly
for name in ["accuracy", "psi", "mean", "wasserstein"]:
    spec = MetricSpec(name=name)
    mtype = resolve_metric_type(spec)
    print(f"{name:<15} → {mtype.value}")

accuracy        → performance
psi             → drift
mean            → statistics
wasserstein     → drift


## 3. Performance metrics

In [5]:
from ayn_ml.core.schema import TabularSchema

schema = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_pred_proba",
    model_id_col=None,
    model_version_col=None,
)

results = {}
for name in ["accuracy", "precision", "recall", "f1", "log_loss", "auc"]:
    spec = MetricSpec(name=name)
    metric = get_metric(name)
    result = metric.compute(current=df_cur, reference=None, schema=schema, spec=spec)
    results[name] = result.value
    print(f"{name:<12}  {result.value:.4f}")

accuracy      0.5167


precision     0.5235
recall        0.5167
f1            0.5170
log_loss      0.8285
auc           0.5195


## 4. Drift metrics

In [6]:
# --- Distribution distance metrics (no effect size) ---
distance_metrics = [
    ("psi", "age", {"n_bins": 10}),  # numeric PSI — union bins, eps=1e-4 default
    ("psi", "region", {}),  # categorical PSI
    ("wasserstein", "age", {}),
    ("mmd", "age", {}),
]

for metric_name, feature, params in distance_metrics:
    spec = MetricSpec(name=metric_name, feature_name=feature, params=params, threshold=0.2)
    result = get_metric(metric_name).compute(current=df_cur, reference=df_ref, schema=schema, spec=spec)
    status = "PASS" if result.status else "FAIL"
    print(f"{metric_name:<14} feature={feature:<8}  value={result.value:.4f}  {status}")

psi 'age': 41 current value(s) outside the reference range [14.84, 64.72]. PSI includes these values (union bins) but their contribution is sensitive to eps=0.0001 — PSI varies 1.7–3.1 for that bin alone at 30% out-of-range density. Cross-check with wasserstein or ks_2samp.


psi            feature=age       value=1.1811  FAIL
psi            feature=region    value=3.4773  FAIL
wasserstein    feature=age       value=9.9668  FAIL
mmd            feature=age       value=0.3762  FAIL


### 4b. Statistical tests + effect sizes

Statistical tests return the **p-value** as `value`.
`effect_size` and `effect_size_label` are set alongside to disambiguate the scale.

Combine both to avoid false alerts on large samples:
- low p-value + small effect → statistically significant but operationally negligible
- low p-value + large effect → genuine drift, alert

In [7]:
# --- Statistical tests (numeric) ---
test_metrics = [
    ("ks_2samp", "age", {}),
    ("ttest", "age", {}),  # Welch's (equal_var=False) by default
    ("ttest", "age", {"equal_var": True}),  # Student's — use only when variances are equal
    ("mannwhitney", "age", {}),  # non-parametric; robust to non-normality
    ("levene", "age", {}),  # tests variance equality
    ("cramervonmises", "income", {}),
    ("anderson_darling", "age", {}),  # tail-sensitive k-sample test
    ("epps_singleton", "age", {}),  # frequency-domain test
]

for metric_name, feature, params in test_metrics:
    spec = MetricSpec(name=metric_name, feature_name=feature, params=params, threshold=0.05)
    result = get_metric(metric_name).compute(current=df_cur, reference=df_ref, schema=schema, spec=spec)
    es = f"{result.effect_size:.3f} ({result.effect_size_label})" if result.effect_size is not None else "—"
    label = params.get("equal_var", False) and " [Student]" or ""
    print(f"{metric_name + label:<22}  p={result.value:.4f}  effect={es}")

print()
# Combined alerting rule: significant p-value AND meaningful effect size
r = get_metric("ttest").compute(
    current=df_cur, reference=df_ref, schema=schema, spec=MetricSpec(name="ttest", feature_name="age", threshold=0.05)
)
drift = r.value < 0.05
material = abs(r.effect_size) >= 0.2  # Cohen's d: <0.2 small, <0.5 medium, >=0.8 large
print(f"age t-test  →  drift={drift}  material={material}  → alert={drift and material}")

### 4c. Chi-square test for categorical features

Chi-square test of homogeneity detects distribution shifts in categorical features.
- `value` = two-tailed p-value
- `effect_size` = Cramér's V ∈ [0, 1] — standardised effect size independent of sample size
- `effect_size_label` = `"cramers_v"`

**Threshold convention:** `upper_bound=False, threshold=0.05` → `status=True` when `p ≥ 0.05` (no drift).

Cramér's V rule of thumb: < 0.1 negligible · < 0.3 moderate · ≥ 0.3 strong.
Combine with the p-value the same way as numeric tests — large samples make any shift significant.

In [8]:
# Chi-square test of homogeneity — categorical features
# reference: region ∈ {A, B, C}   current: region ∈ {A, B, C, D}  → new category introduced

spec_chi = MetricSpec(
    name="chisquare",
    feature_name="region",
    threshold=0.05,
    upper_bound=False,  # status=True when p >= 0.05 (no drift)
)

# Current window has new category 'D' → distribution shifted
r_drift = get_metric("chisquare").compute(current=df_cur, reference=df_ref, schema=schema, spec=spec_chi)
print(
    f"region (cur adds 'D')  p={r_drift.value:.4f}  V={r_drift.effect_size:.3f}"
    f"  ({r_drift.effect_size_label})  {'PASS' if r_drift.status else 'FAIL'}"
)

# Identical distributions — baseline, no drift expected
r_stable = get_metric("chisquare").compute(current=df_ref, reference=df_ref, schema=schema, spec=spec_chi)
print(
    f"region (identical)     p={r_stable.value:.4f}  V={r_stable.effect_size:.3f}"
    f"  ({r_stable.effect_size_label})  {'PASS' if r_stable.status else 'FAIL'}"
)

# Combined alert: flag only when BOTH p-value is small AND effect is meaningful
drift = r_drift.value < 0.05
material = r_drift.effect_size >= 0.1  # V < 0.1 → negligible even if significant
print(f"\nregion  drift={drift}  material={material}  → alert={drift and material}")

region (cur adds 'D')  p=0.0000  V=0.340  (cramer_v)  FAIL
region (identical)     p=1.0000  V=0.000  (cramer_v)  PASS

region  drift=True  material=True  → alert=True


In [ ]:
# --- New drift tests: categorical/binary and distance metrics ---

# G-test (log-likelihood ratio) — categorical/binary, Cramér's V as effect size
spec_gtest = MetricSpec(name="gtest", feature_name="region", threshold=0.05)
r_g = get_metric("gtest").compute(current=df_cur, reference=df_ref, schema=schema, spec=spec_gtest)
print(f"gtest          region  p={r_g.value:.4f}  V={r_g.effect_size:.3f} ({r_g.effect_size_label})")

# Binary column: y_true (0/1) — Fisher exact and z-test proportions
spec_fisher = MetricSpec(name="fisher_exact", feature_name="y_true", threshold=0.05)
r_f = get_metric("fisher_exact").compute(current=df_cur, reference=df_ref, schema=schema, spec=spec_fisher)
print(f"fisher_exact   y_true  p={r_f.value:.4f}  odds_ratio={r_f.effect_size:.3f} ({r_f.effect_size_label})")

spec_z = MetricSpec(name="ztest_proportions", feature_name="y_true", threshold=0.05)
r_z = get_metric("ztest_proportions").compute(current=df_cur, reference=df_ref, schema=schema, spec=spec_z)
print(f"ztest_props    y_true  p={r_z.value:.4f}  z={r_z.effect_size:.3f} ({r_z.effect_size_label})")

print()
# --- Distance-based drift metrics (upper-bound threshold: alert when value > threshold) ---
# hellinger, jensenshannon, tvd: accept numeric and categorical
# energy_distance: numeric/binary only
distance_drift = [
    ("hellinger",      "age",    0.10),  # numeric, histogram-based
    ("hellinger",      "region", 0.10),  # categorical, category counts
    ("jensenshannon",  "age",    0.10),
    ("tvd",            "age",    0.10),
    ("energy_distance","age",    5.0),   # no upper bound — domain-dependent threshold
]

for metric_name, feature, thr in distance_drift:
    spec = MetricSpec(name=metric_name, feature_name=feature, threshold=thr, upper_bound=True)
    result = get_metric(metric_name).compute(current=df_cur, reference=df_ref, schema=schema, spec=spec)
    status = "PASS" if result.status else "DRIFT"
    print(f"{metric_name:<16} feature={feature:<8}  value={result.value:.4f}  {status}")

### 4d. Target drift — detecting P(Y) shifts

`target_drift` monitors shifts in the **label distribution** P(Y) between a reference and current window.
It uses PSI internally with automatic type detection:
- **Integer labels** → treated as categorical (frequency counts, not histogram bins)
- **Float labels** → treated as numeric (histogram binning)
- **`params["treat_as"]`** → `"numeric"` or `"categorical"` to override

Value is the PSI score (same thresholds as `psi`: < 0.1 stable · < 0.25 moderate · ≥ 0.25 significant).

> **Blind spot:** `target_drift` only detects P(Y) shifts (class imbalance changes).
> Silent concept drift — where P(Y|X) changes but P(Y) stays the same — requires pairing with
> feature drift metrics such as `psi` or `wasserstein`.

In [9]:
# --- Binary classification: detect class imbalance drift ---
# reference: balanced 50/50   current: imbalanced 30/70

rng2 = np.random.default_rng(42)
n2 = 300

df_ref_td = pd.DataFrame({"y_true": rng2.choice([0, 1], size=n2, p=[0.5, 0.5])})
df_cur_drifted = pd.DataFrame({"y_true": rng2.choice([0, 1], size=n2, p=[0.3, 0.7])})
df_cur_stable = pd.DataFrame({"y_true": rng2.choice([0, 1], size=n2, p=[0.5, 0.5])})

schema_td = TabularSchema(label_col="y_true", model_id_col=None, model_version_col=None)
spec_td = MetricSpec(name="target_drift", threshold=0.1)

r_drifted = get_metric("target_drift").compute(df_cur_drifted, df_ref_td, schema_td, spec_td)
r_stable = get_metric("target_drift").compute(df_cur_stable, df_ref_td, schema_td, spec_td)

print(f"drifted 30/70   PSI={r_drifted.value:.4f}  {'PASS' if r_drifted.status else 'FAIL'}")
print(f"stable  50/50   PSI={r_stable.value:.4f}   {'PASS' if r_stable.status else 'FAIL'}")

drifted 30/70   PSI=0.2393  FAIL
stable  50/50   PSI=0.0011   PASS


In [10]:
# --- Multiclass: string labels, shifted proportions ---
# Integers default to categorical PSI; strings always categorical.
# Use treat_as='numeric' to force histogram binning (e.g. ordinal ratings 1–5).

df_ref_mc = pd.DataFrame({"y_true": rng2.choice(["cat", "dog", "bird"], size=n2, p=[0.5, 0.3, 0.2])})
df_cur_mc = pd.DataFrame(
    {
        "y_true": rng2.choice(["cat", "dog", "bird"], size=n2, p=[0.2, 0.5, 0.3])  # shifted
    }
)

r_mc = get_metric("target_drift").compute(df_cur_mc, df_ref_mc, schema_td, spec_td)
print(f"multiclass shift  PSI={r_mc.value:.4f}  {'PASS' if r_mc.status else 'FAIL'}")

# Force numeric binning — useful for ordinal labels (e.g. 1–5 ratings, 0–10 scores)
df_ref_ord = pd.DataFrame({"y_true": rng2.integers(1, 6, size=n2).astype(float)})
df_cur_ord = pd.DataFrame({"y_true": rng2.integers(3, 8, size=n2).astype(float)})  # shifted up

spec_numeric = MetricSpec(name="target_drift", threshold=0.1, params={"treat_as": "numeric"})
r_ord = get_metric("target_drift").compute(df_cur_ord, df_ref_ord, schema_td, spec_numeric)
print(f"ordinal 1-5→3-7 (numeric PSI)  PSI={r_ord.value:.4f}  {'PASS' if r_ord.status else 'FAIL'}")

target_drift 'y_true': 125 current value(s) outside the reference range [1, 5]. PSI includes these values (union bins) but their contribution is sensitive to eps=0.0001 — PSI varies 1.7–3.1 for that bin alone at 30% out-of-range density. Cross-check with wasserstein or ks_2samp.


multiclass shift  PSI=0.4122  FAIL
ordinal 1-5→3-7 (numeric PSI)  PSI=6.3385  FAIL


## 5. Statistics metrics

In [11]:
# --- Column-level statistics ---
stats = [
    ("mean", "age", {}),
    ("std", "age", {}),
    ("median", "income", {}),
    ("quantile", "age", {"q": 0.95}),
    ("skewness", "income", {}),
    ("top_category", "region", {}),
    ("count", "age", {}),
    ("sum", "income", {}),
    ("unique_count", "region", {}),
    ("in_range_count", "age", {"low": 30.0, "high": 60.0}),
    ("out_range_count", "age", {"low": 30.0, "high": 60.0}),
    ("in_list_count", "region", {"values": ["A", "B"]}),
]

for metric_name, feature, params in stats:
    spec = MetricSpec(name=metric_name, feature_name=feature, params=params)
    metric = get_metric(metric_name)
    result = metric.compute(current=df_cur, reference=None, schema=schema, spec=spec)
    print(f"{metric_name:<18} feature={feature:<8}  value={result.value}")

print()
# --- Dataset-level statistics (feature_name=None) ---
dataset_stats = [
    ("row_count", {}),
    ("column_count", {}),
    ("almost_constant_columns", {"n_unique": 1}),  # truly constant columns only
    ("duplicate_rows", {}),
    ("empty_columns", {}),
]

for metric_name, params in dataset_stats:
    spec = MetricSpec(name=metric_name, params=params)
    metric = get_metric(metric_name)
    result = metric.compute(current=df_cur, reference=None, schema=schema, spec=spec)
    print(f"{metric_name:<25}  value={result.value}")

## 6. Threshold evaluation with `compute_status`

In [12]:
from ayn_ml.metrics.base import compute_status

examples = [
    # (value, threshold, upper_bound)  — upper_bound=True means pass when value <= threshold
    (0.88, 0.85, False),  # accuracy >= 0.85  → PASS
    (0.72, 0.85, False),  # accuracy >= 0.85  → FAIL
    (0.15, 0.2, True),  # psi      <= 0.20  → PASS
    (0.31, 0.2, True),  # psi      <= 0.20  → FAIL
    (0.50, [0.3, 0.7], True),  # value in [0.3, 0.7] → PASS
    (0.25, [0.3, 0.7], True),  # value in [0.3, 0.7] → FAIL
]

for value, threshold, upper_bound in examples:
    spec = MetricSpec(name="x", threshold=threshold, upper_bound=upper_bound)
    status = compute_status(value, spec)
    label = "PASS" if status else "FAIL"
    print(f"value={value}  threshold={threshold}  upper_bound={upper_bound}  → {label}")

value=0.88  threshold=0.85  upper_bound=False  → PASS
value=0.72  threshold=0.85  upper_bound=False  → FAIL
value=0.15  threshold=0.2  upper_bound=True  → PASS
value=0.31  threshold=0.2  upper_bound=True  → FAIL
value=0.5  threshold=[0.3, 0.7]  upper_bound=True  → PASS
value=0.25  threshold=[0.3, 0.7]  upper_bound=True  → FAIL
